# 06 · Silver: del texto METAR a columnas y al clima de cada hora

**Propósito:** convertir el dato **no estructurado** del proyecto (el texto libre de cada METAR) en columnas
numéricas y banderas, y armar una tabla con el clima conocido en cada aeropuerto a cada hora.

**Entrada:** `bronze/metar/`.

**Salida:**
- `silver/metar_obs/year=/month=/`: una fila por reporte METAR, ya decodificado.
- `silver/clima_horario/year=/month=/`: una fila por aeropuerto y hora `k`, con la **última observación
  estrictamente anterior a `k`**. Así, al unir un vuelo con la hora `k`, solo se usa clima que ya se conocía.
- `docs/evidencias/silver/metar_campos_parseados.csv`: qué porcentaje de reportes trae cada campo.

**Requisito:** notebook 03.

El decodificado usa expresiones regulares nativas de Spark (`regexp_extract`), sin funciones de Python fila por
fila: corre dentro de la JVM y escala con el clúster.

In [ ]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import pandas as pd

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-silver-clima")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

In [ ]:
EVID = RAIZ / "docs" / "evidencias" / "silver"
EVID.mkdir(parents=True, exist_ok=True)
FFILL_HORAS = cfg["clima"]["ffill_max_horas"]

esquema_metar = T.StructType([
    T.StructField("estacion_iem", T.StringType()),
    T.StructField("valid", T.StringType()),
    T.StructField("metar", T.StringType()),
])
metar = (
    spark.read.option("header", False).schema(esquema_metar).csv(ruta("bronze_metar"))
    .where(F.col("valid") != "valid")  # descarta la fila de encabezado de cada archivo
    .withColumnRenamed("station", "aeropuerto")  # sale de la carpeta station=<aeropuerto>
    .withColumn("obs_utc", F.to_timestamp("valid", "yyyy-MM-dd HH:mm"))
    .where(F.col("obs_utc").isNotNull() & F.col("metar").isNotNull())
    .where((F.date_format("obs_utc", "yyyy-MM") >= DESDE) & (F.date_format("obs_utc", "yyyy-MM") <= HASTA))
)

## 1. Preparar el texto

- Se quita el prefijo `METAR` o `SPECI` y todo lo que viene después de `RMK` (observaciones internas).
- Lo que viene después de `TEMPO` o `BECMG` es un pronóstico, no lo observado: se marca `tiene_tempo` y se descarta
  para el resto del decodificado.
- Los campos de visibilidad, nubes, temperatura, presión y fenómenos se buscan **después del grupo de viento**, para
  no confundir el código de la estación (por ejemplo `KSNA`) con un fenómeno (`SN`, nieve).

In [ ]:
sin_rmk = F.regexp_replace(F.regexp_replace("metar", r"^(METAR|SPECI) ", ""), r" RMK .*$", "")
metar = metar.withColumn("tiene_tempo", sin_rmk.rlike(r" (TEMPO|BECMG) ").cast("int"))
metar = metar.withColumn("principal", F.regexp_replace(sin_rmk, r" (TEMPO|BECMG) .*$", ""))
despues_viento = F.regexp_extract("principal", r"(?:KT|MPS) (.*)$", 1)
metar = metar.withColumn("cuerpo", F.when(despues_viento != "", despues_viento).otherwise(F.col("principal")))

## 2. Decodificar

| Campo | Patrón | Ejemplo |
|---|---|---|
| Viento | `ddd` grados + `ff` velocidad + `Gff` ráfaga + unidad (`KT` nudos o `MPS` m/s) | `12009KT`, `VRB03KT`, `27015G25KT` |
| Visibilidad en metros | 4 dígitos (`9999` = 10 km o más) | `9999`, `0800` |
| Visibilidad en millas | entero, fracción o ambos, con prefijo `M` (menos de) o `P` (más de) | `10SM`, `1/2SM`, `1 1/2SM`, `P6SM` |
| Techo | la capa `BKN`, `OVC` o `VV` más baja, en cientos de pies | `BKN013` = 1300 pies |
| Temperatura y rocío | `tt/dd`, con `M` para bajo cero | `19/17`, `M02/M05` |
| Presión | `Qpppp` en hPa o `Apppp` en centésimas de inHg | `Q1018`, `A3006` |
| Fenómenos | códigos de tiempo presente | `FG`, `BR`, `TSRA`, `-RA`, `SN` |

`CAVOK` significa visibilidad de 10 km o más y sin nubes bajas. Si un reporte se pudo leer pero no tiene capas
`BKN`, `OVC` ni `VV`, no hay techo: se guarda como 99 999 pies para que la categoría de vuelo lo trate como cielo
despejado.

In [ ]:
def grupo(columna, patron, i):
    valor = F.regexp_extract(columna, patron, i)
    return F.when(valor != "", valor)


# Viento
PATRON_VIENTO = r" (\d{3}|VRB)(\d{2,3})(?:G(\d{2,3}))?(KT|MPS)"
a_nudos = F.when(grupo("principal", PATRON_VIENTO, 4) == "MPS", 1.94384).otherwise(1.0)
metar = (
    metar.withColumn("viento_dir", grupo("principal", PATRON_VIENTO, 1).cast("int"))
    .withColumn("viento_kt", F.round(grupo("principal", PATRON_VIENTO, 2).cast("double") * a_nudos, 1))
    .withColumn("rafaga_kt", F.round(grupo("principal", PATRON_VIENTO, 3).cast("double") * a_nudos, 1))
)

# Visibilidad: CAVOK, metros (Brasil) o millas terrestres con fracciones (EE. UU.)
cavok = F.col("cuerpo").rlike(r"(^| )CAVOK( |$)")
vis_metros = grupo("cuerpo", r"^(?:\d{3}V\d{3} )?(\d{4})(?:NDV)?(?= |$)", 1).cast("int")
PATRON_SM = r"(?:^| )(M|P)?(?:(\d{1,2}) )?(?:(\d{1,2})/(\d{1,2})|(\d{1,2}))SM(?= |$)"
entero_solo = grupo("cuerpo", PATRON_SM, 5).cast("double")
entero = F.coalesce(grupo("cuerpo", PATRON_SM, 2).cast("double"), F.lit(0.0))
fraccion = grupo("cuerpo", PATRON_SM, 3).cast("double") / grupo("cuerpo", PATRON_SM, 4).cast("double")
vis_millas = F.coalesce(entero_solo, entero + fraccion)
metar = metar.withColumn(
    "visibilidad_m",
    F.when(cavok, 10000.0)
    .when(vis_metros == 9999, 10000.0)
    .when(vis_metros.isNotNull(), vis_metros.cast("double"))
    .otherwise(F.round(vis_millas * 1609.34, 0)),
)

# Techo: la capa BKN/OVC/VV más baja
capas = F.regexp_extract_all("cuerpo", F.lit(r"(?:BKN|OVC|VV)(\d{3})"), 1)
techo = F.array_min(F.transform(capas, lambda x: x.cast("int"))) * 100
se_leyo = F.col("viento_kt").isNotNull() | F.col("visibilidad_m").isNotNull()
metar = metar.withColumn("techo_ft", F.when(F.size(capas) > 0, techo).when(se_leyo | cavok, 99999))

# Temperatura y punto de rocío (M = bajo cero)
PATRON_TEMP = r"(?:^| )(M?\d{2})/(M?\d{2})?(?= |$)"
metar = (
    metar.withColumn("temp_c", F.regexp_replace(grupo("cuerpo", PATRON_TEMP, 1), "M", "-").cast("int"))
    .withColumn("rocio_c", F.regexp_replace(grupo("cuerpo", PATRON_TEMP, 2), "M", "-").cast("int"))
    .withColumn("spread_t_td", F.col("temp_c") - F.col("rocio_c"))
)

# Presión: Q en hPa o A en centésimas de pulgada de mercurio
q = grupo("cuerpo", r"(?:^| )Q(\d{4})(?= |$)", 1).cast("double")
a = grupo("cuerpo", r"(?:^| )A(\d{4})(?= |$)", 1).cast("double")
metar = metar.withColumn("presion_hpa", F.round(F.coalesce(q, a / 100 * 33.8639), 1))

# Fenómenos
FENOMENOS = {
    "fen_niebla": r"(^| )(VC|MI|BC|PR|FZ)?FG( |$)",
    "fen_neblina": r"(^| )BR( |$)",
    "fen_tormenta": r"(^| )[-+]?(VC)?TS[A-Z]*( |$)",
    "fen_lluvia": r"(^| )[-+]?(VC)?(SH|TS|FZ)?(RA|DZ)[A-Z]*( |$)",
    "fen_nieve": r"(^| )[-+]?(VC|BL|DR|SH)?([A-Z]{2})?SN( |$)",
}
for nombre, patron in FENOMENOS.items():
    metar = metar.withColumn(nombre, F.col("cuerpo").rlike(patron).cast("int"))

# Categoría de vuelo: la peor entre techo y visibilidad
millas = F.col("visibilidad_m") / 1609.34
metar = metar.withColumn(
    "categoria_vuelo",
    F.when((F.col("techo_ft") < 500) | (millas < 1), "LIFR")
    .when((F.col("techo_ft") < 1000) | (millas < 3), "IFR")
    .when((F.col("techo_ft") <= 3000) | (millas <= 5), "MVFR")
    .when(F.col("techo_ft").isNotNull() & F.col("visibilidad_m").isNotNull(), "VFR"),
)

## 3. Cinco reportes reales y su decodificado

In [ ]:
CAMPOS = [
    "viento_dir", "viento_kt", "rafaga_kt", "visibilidad_m", "techo_ft", "temp_c", "rocio_c", "presion_hpa",
    "fen_niebla", "fen_neblina", "fen_tormenta", "fen_lluvia", "fen_nieve", "tiene_tempo", "categoria_vuelo",
]
ejemplos = []
for condicion in [
    F.col("metar").rlike(r" 9999 ") & F.col("metar").contains("Q"),
    F.col("metar").rlike(r" \d+ \d/\dSM "),
    F.col("metar").rlike(r" [-+]?TS"),
    F.col("metar").rlike(r" (BC|MI|FZ)?FG "),
    F.col("metar").rlike(r" (BKN|OVC)00\d "),
]:
    ejemplos += metar.where(condicion).select("aeropuerto", "metar", *CAMPOS).limit(1).collect()
pd.DataFrame([e.asDict() for e in ejemplos]).T

## 4. Deduplicación y escritura de `metar_obs`

IEM a veces agrega reportes armados a partir de otra fuente (terminan en `IEM_GHCNH`). Si en el mismo minuto hay un
METAR original, se prefiere el original.

In [ ]:
orden = Window.partitionBy("aeropuerto", "obs_utc").orderBy(F.col("metar").contains("IEM_GHCNH").asc())
metar_obs = (
    metar.withColumn("_n", F.row_number().over(orden)).where("_n = 1")
    .select("aeropuerto", "estacion_iem", "obs_utc", "metar", "spread_t_td", *CAMPOS)
    .withColumn("year", F.year("obs_utc")).withColumn("month", F.month("obs_utc"))
)
(
    metar_obs.repartition("year", "month")
    .write.mode("overwrite").partitionBy("year", "month").parquet(ruta("silver_metar"))
)
metar_obs = spark.read.parquet(ruta("silver_metar"))
total = metar_obs.count()
print(f"silver/metar_obs: {total:,} reportes de {metar_obs.select('aeropuerto').distinct().count()} aeropuertos")

In [ ]:
# Porcentaje de reportes en que se pudo leer cada campo
parseo = metar_obs.select([
    F.round(100 * F.count(c) / F.count("*"), 2).alias(c)
    for c in ["viento_kt", "visibilidad_m", "techo_ft", "temp_c", "rocio_c", "presion_hpa", "categoria_vuelo"]
]).toPandas().T.rename(columns={0: "pct_con_dato"})
parseo.to_csv(EVID / "metar_campos_parseados.csv", index_label="campo")
parseo

## 5. Clima de cada hora (`clima_horario`)

Para cada aeropuerto y cada hora `k` se busca la última observación **estrictamente anterior a `k`**:

1. Cada observación se asigna a la hora siguiente a su minuto: un METAR de las 13:53 pertenece a `k` = 14:00. Si hay
   varias para la misma `k`, se queda la más reciente.
2. Se arma la grilla completa de horas de cada aeropuerto con `sequence` (sin cruzar tablas), y la observación se
   arrastra hacia adelante como máximo `ffill_max_horas` horas. Pasado ese tiempo el dato se considera viejo y queda
   vacío.
3. Se recuperan las columnas de esa misma observación (no se mezclan campos de reportes distintos) y se calcula la
   tendencia de presión de las últimas 3 horas.

In [ ]:
UNA_HORA = F.expr("INTERVAL 1 HOUR")
por_hora = (
    metar_obs.withColumn("k", F.date_trunc("hour", "obs_utc") + UNA_HORA)
    .groupBy("aeropuerto", "k").agg(F.max("obs_utc").alias("obs_utc"))
)
rangos = por_hora.groupBy("aeropuerto").agg(F.min("k").alias("desde"), F.max("k").alias("hasta"))
grilla = rangos.select(
    "aeropuerto",
    F.explode(F.sequence("desde", F.col("hasta") + F.expr(f"INTERVAL {FFILL_HORAS} HOURS"), UNA_HORA)).alias("k"),
)
ventana = Window.partitionBy("aeropuerto").orderBy("k").rowsBetween(-FFILL_HORAS, 0)
vigente = (
    grilla.join(por_hora, ["aeropuerto", "k"], "left")
    .withColumn("obs_utc", F.last("obs_utc", ignorenulls=True).over(ventana))
    .where(F.col("obs_utc").isNotNull())
)
clima = vigente.join(metar_obs.drop("year", "month", "metar", "estacion_iem"), ["aeropuerto", "obs_utc"], "left")
hace_3h = clima.select(
    "aeropuerto", (F.col("k") + F.expr("INTERVAL 3 HOURS")).alias("k"), F.col("presion_hpa").alias("presion_3h")
)
clima_horario = (
    clima.join(hace_3h, ["aeropuerto", "k"], "left")
    .withColumn("presion_tendencia_3h", F.round(F.col("presion_hpa") - F.col("presion_3h"), 1))
    .withColumn("horas_desde_obs", F.round((F.unix_timestamp("k") - F.unix_timestamp("obs_utc")) / 3600, 2))
    .drop("presion_3h")
    .withColumn("year", F.year("k")).withColumn("month", F.month("k"))
)
(
    clima_horario.repartition("year", "month")
    .write.mode("overwrite").partitionBy("year", "month").parquet(ruta("silver_clima_horario"))
)

## 6. Verificación

In [ ]:
clima_horario = spark.read.parquet(ruta("silver_clima_horario"))
futuras = clima_horario.where(F.col("obs_utc") >= F.col("k")).count()
assert futuras == 0, f"{futuras} horas usan una observación que no es anterior a k"
print(f"silver/clima_horario: {clima_horario.count():,} filas (aeropuerto × hora)")
print("Ninguna hora usa una observación igual o posterior a k.")
resumen = clima_horario.agg(
    F.round(F.avg("horas_desde_obs"), 2).alias("horas_desde_obs_media"),
    F.round(100 * F.avg((F.col("horas_desde_obs") > 1).cast("int")), 2).alias("pct_rellenadas"),
    F.round(100 * F.count("presion_tendencia_3h") / F.count("*"), 2).alias("pct_con_tendencia"),
).toPandas()
resumen.to_csv(EVID / "clima_horario_resumen.csv", index=False)
resumen

In [ ]:
clima_horario.where(F.col("aeropuerto") == "SBGR").orderBy("k").select(
    "k", "obs_utc", "horas_desde_obs", "viento_kt", "visibilidad_m", "techo_ft", "categoria_vuelo",
    "presion_hpa", "presion_tendencia_3h",
).limit(8).toPandas()

In [ ]:
spark.stop()